# New fold seeds for the nested ask-rule evaluation

The paper's cost-aware ask rule was evaluated on one assignment of rows to five folds. This run repeats the nested
evaluation on five new assignments. For each one it fits the decision stage on folds 0–2 only (525 fits) and the
evaluation stage on folds 3–4 (350 fits). Every fit is saved to Drive as soon as it finishes. After a disconnect, run
cells 1 and 3 again and then the cell that was running: finished fits are skipped.

**scikit-learn 1.9.1 is required.** The original fits used it; Colab's default version gives different fits. Cell 2
installs it and cell 3 refuses to continue with any other version.

**Runtime:** CPU (the default) is enough. About 1 min for cell 2, about 25 min for cell 5 and about 1 hour per seed in
cell 6, so about 5 hours in all. It can be split over several sessions.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/task_inference_seeds'   # change if you uploaded elsewhere
RESULTS = BASE + '/results'
import os
for f in ('config.json', 'targets.csv', 'frozen_seed0.csv', 'nested_lib.py', 'seeds_lib.py', 'data'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f}'
print('ok')

## 2. Install scikit-learn 1.9.1 (~1 min)

If Colab asks you to restart the session after this, click **Restart**, then run cell 1 and continue with cell 3.

In [ ]:
%pip install -q "scikit-learn==1.9.1"

## 3. Load the library and check the version and the tables

This cell stops with a message if scikit-learn is not 1.9.1. Then use Runtime › Restart session and run cell 1 and this
cell again. It also checks every table against its checksum.

In [ ]:
import sys, importlib
sys.path.insert(0, BASE)
import nested_lib, seeds_lib
importlib.reload(nested_lib); importlib.reload(seeds_lib)
cfg, targets = nested_lib.load_bundle(BASE)
nested_lib.check_sklearn(cfg)
print(seeds_lib.verify_data(BASE), 'tables match their checksums')
print(cfg)
print(nested_lib.versions())

## 4. Smoke test (about a minute, writes to /content)

I010 is a target whose fits depend on the scikit-learn version. The output must say `evaluation rows in any nested
training set: 0` and `10/10 identical`. If it does not, stop and send the output.

In [ ]:
seeds_lib.run_check(BASE, '/content/smoke_seeds', only={'I010'})
seeds_lib.run(BASE, '/content/smoke_seeds', 1, only={'I010'})
_ = seeds_lib.summarize('/content/smoke_seeds', BASE)

## 5. Check against the frozen fits (~25 min, resumable)

Seed 0 is not a result. It refits one nested fold and one evaluation fold of the original assignment, to confirm that
this environment reproduces the original fits. The largest target (isolet) comes last and takes several minutes.

In [ ]:
seeds_lib.run_check(BASE, RESULTS)

## 6. Fold seeds 1 to 5 (~1 hour per seed, resumable)

The seeds run in order. A disconnect loses at most the fit that was running.

In [ ]:
seeds_lib.run_seeds(BASE, RESULTS)

## 7. Summary and zip

Each seed should show `complete True`. The check should say `350/350 identical` (the October check had one fit out of 175
that differed slightly, so 348 or 349 is fine) and `evaluation rows in any nested training set: 0`. Download
`seeds_results.zip` (also saved in the Drive folder) and put it in the project folder.

In [ ]:
import shutil
_ = seeds_lib.summarize(RESULTS, BASE)
zip_path = shutil.make_archive(BASE + '/seeds_results', 'zip', RESULTS)
print('wrote', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:  # the zip is also in the Drive folder
    print(f'automatic download failed ({e}); download seeds_results.zip from the Drive folder')

### Optional: retry fits that failed

Only if cell 7 lists errors that look like a disconnect or out-of-memory. Then run cell 7 again.

In [ ]:
for s in cfg['fold_seeds']:
    seeds_lib.run(BASE, RESULTS, s, retry_errors=True)